# 11 · Data quality checks and reconciliation (Oracle)

A pipeline must **prove** its data is right. These SQL checks are the same ones you put in a
data quality step (in Composer/Airflow, dbt tests, or Great Expectations).

Good checks return **a number you can compare**: 0 bad rows = pass.

- completeness: NULLs where there should be none
- uniqueness: duplicate keys
- referential integrity: child rows whose parent is missing (**orphans**)
- validity: values outside the allowed range/list
- **reconciliation**: source and target have the same row count and the same SUM

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/11_data_quality.ipynb`](../../oracle/practice/11_data_quality.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. NULL profile of a table  ·  🟢 easy

For `orders`, in one row show `total_rows` and the number of NULLs in each column: `null_order_id`, `null_customer_id`, `null_order_date`, `null_amount`, `null_status`.

*Expected result: 1 row · columns: `total_rows`, `null_order_id`, `null_customer_id`, `null_order_date`, `null_amount`, `null_status`*

**How it works:** For each column, `COUNT(*) − COUNT(col)` = NULLs. All columns in one SELECT = one table scan.

In [ ]:
%%oracle
SELECT COUNT(*)                     AS total_rows,
       COUNT(*) - COUNT(order_id)    AS null_order_id,
       COUNT(*) - COUNT(customer_id) AS null_customer_id,
       COUNT(*) - COUNT(order_date)  AS null_order_date,
       COUNT(*) - COUNT(amount)      AS null_amount,
       COUNT(*) - COUNT(status)      AS null_status
FROM orders

,total_rows,null_order_id,null_customer_id,null_order_date,null_amount,null_status
0,108,0,0,0,6,0


**Why / traps:** `COUNT(*) - COUNT(col)` = number of NULLs. One scan for all columns.

## Q2. Is the key unique?  ·  🟢 easy

Check if `order_id` is a unique key in `orders`. Show `dup_keys` = how many order_ids appear more than once, and `extra_rows` = how many rows too many.

*Expected result: 1 row · columns: `dup_keys`, `extra_rows`*

**How it works:** Group by `order_id` and keep keys with more than 1 row. Count those keys, and sum `copies − 1` for the extra rows.

In [ ]:
%%oracle
SELECT COUNT(*)            AS dup_keys,
       SUM(copies - 1)     AS extra_rows
FROM (SELECT order_id, COUNT(*) AS copies
      FROM orders
      GROUP BY order_id
      HAVING COUNT(*) > 1) d

,dup_keys,extra_rows
0,4,4


**Why / traps:** Pass = 0. Faster check: `COUNT(*) = COUNT(DISTINCT order_id)`.

## Q3. Orphan rows (referential integrity)  ·  🟡 medium

Make one test-report query with UNION ALL. Rows: `check_name`, `bad_rows`:
- `items_without_order`: order_items whose order_id is not in orders
- `items_without_product`: order_items whose product_id is not in products
- `employee_bad_dept`: employees whose dept_id is **not NULL** but not in departments
- `employee_bad_manager`: employees whose manager_id is not NULL but not an emp_id

*Expected result: 4 rows · columns: `check_name`, `bad_rows`*

**How it works:** One small query per rule, each `COUNT(*)` of rows with no parent (NOT EXISTS), stacked with UNION ALL into a report.

In [ ]:
%%oracle
SELECT 'items_without_order' AS check_name, COUNT(*) AS bad_rows
FROM order_items i
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.order_id = i.order_id)
UNION ALL
SELECT 'items_without_product', COUNT(*)
FROM order_items i
WHERE NOT EXISTS (SELECT 1 FROM products p WHERE p.product_id = i.product_id)
UNION ALL
SELECT 'employee_bad_dept', COUNT(*)
FROM employees e
WHERE e.dept_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM departments d WHERE d.dept_id = e.dept_id)
UNION ALL
SELECT 'employee_bad_manager', COUNT(*)
FROM employees e
WHERE e.manager_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM employees m WHERE m.emp_id = e.manager_id)

,check_name,bad_rows
0,items_without_order,0
1,items_without_product,0
2,employee_bad_dept,0
3,employee_bad_manager,0


**Why / traps:** A NULL foreign key is **not** an orphan (it means "no parent"). Data warehouses like BigQuery do **not enforce** foreign keys, so you must test them yourself.

## Q4. Validity rules  ·  🟡 medium

Count rows of `orders` that break each rule. One row with: `negative_amount` (amount < 0), `huge_amount` (amount > 1000), `future_date` (order_date after 2025-12-31), `bad_status` (status not in completed / cancelled / pending / returned, or NULL).

*Expected result: 1 row · columns: `negative_amount`, `huge_amount`, `future_date`, `bad_status`*

**How it works:** One `SUM(CASE WHEN rule broken THEN 1 ELSE 0 END)` per rule, all in one scan.

In [ ]:
%%oracle
SELECT SUM(CASE WHEN amount < 0 THEN 1 ELSE 0 END)                         AS negative_amount,
       SUM(CASE WHEN amount > 1000 THEN 1 ELSE 0 END)                      AS huge_amount,
       SUM(CASE WHEN order_date > DATE '2025-12-31' THEN 1 ELSE 0 END)     AS future_date,
       SUM(CASE WHEN status IS NULL
                  OR status NOT IN ('completed', 'cancelled', 'pending', 'returned')
                THEN 1 ELSE 0 END)                                          AS bad_status
FROM orders

,negative_amount,huge_amount,future_date,bad_status
0,0,0,0,0


**Why / traps:** `status NOT IN (...)` is unknown for a NULL status, so NULL must be checked separately.

## Q5. Reconciliation: source vs target  ·  🟡 medium

The target is a daily summary built from completed orders. Compare it with the source:
`src_rows`, `tgt_rows` (number of orders in each), `src_amount`, `tgt_amount`, and `result` = `MATCH` if both pairs are equal, else `MISMATCH`.

*Expected result: 1 row · columns: `src_rows`, `tgt_rows`, `src_amount`, `tgt_amount`, `result`*

**How it works:** Count and sum the source; count and sum the target; compare both pairs in a CASE → MATCH / MISMATCH.

In [ ]:
%%oracle
WITH src AS (
    SELECT DISTINCT order_id, order_date, amount
    FROM orders
    WHERE status = 'completed'
),
tgt AS (   -- the "loaded" table
    SELECT order_date, COUNT(*) AS num_orders, SUM(amount) AS amount
    FROM src
    GROUP BY order_date
),
s AS (SELECT COUNT(*) AS n, SUM(amount) AS amt FROM src),
t AS (SELECT SUM(num_orders) AS n, SUM(amount) AS amt FROM tgt)
SELECT s.n AS src_rows, t.n AS tgt_rows, s.amt AS src_amount, t.amt AS tgt_amount,
       CASE WHEN s.n = t.n AND s.amt = t.amt THEN 'MATCH' ELSE 'MISMATCH' END AS result
FROM s CROSS JOIN t

,src_rows,tgt_rows,src_amount,tgt_amount,result
0,70,70,6115,6115,MATCH


**Why / traps:** Count + sum reconciliation catches lost rows, duplicated rows and changed amounts. In a pipeline, write the result to an audit table and **fail the job** on MISMATCH.

## Q6. Row-level differences between source and target  ·  🔴 hard

Using the `src` and `tgt` CTEs given in the answer cell (tgt is missing order 1005 and has a wrong amount for 1010), list `order_id` and `issue` = `missing_in_target`, `missing_in_source` or `amount_differs`. Careful: some amounts are NULL on both sides: that is **not** a difference.

<details><summary>Hint</summary>

FULL OUTER JOIN on order_id. Then compare amounts with a NULL-safe test.

</details>

*Expected result: 2 rows · columns: `order_id`, `issue`*

**How it works:** FULL OUTER JOIN source and target on the key. Missing side = missing row. Both present but amounts not equal (NULL-safe) = amount differs.

In [ ]:
%%oracle
WITH src AS (
    SELECT DISTINCT order_id, amount FROM orders
),
tgt AS (   -- pretend this is the loaded target: 1005 is missing, 1010 has a wrong amount
    SELECT order_id, CASE WHEN order_id = 1010 THEN amount + 10 ELSE amount END AS amount
    FROM src
    WHERE order_id <> 1005
)
SELECT COALESCE(s.order_id, t.order_id) AS order_id,
       CASE WHEN t.order_id IS NULL THEN 'missing_in_target'
            WHEN s.order_id IS NULL THEN 'missing_in_source'
            ELSE 'amount_differs'
       END AS issue
FROM src s
FULL OUTER JOIN tgt t ON t.order_id = s.order_id
WHERE s.order_id IS NULL
   OR t.order_id IS NULL
   OR NOT (s.amount = t.amount OR (s.amount IS NULL AND t.amount IS NULL))
ORDER BY order_id

,order_id,issue
0,1005,missing_in_target
1,1010,amount_differs


**Why / traps:** `s.amount <> t.amount` is unknown when one side is NULL, so a NULL vs 100 difference would be missed, and NULL vs NULL must count as equal. Engines have helpers: BigQuery `IS DISTINCT FROM`, Spark `<=>` / `eqNullSafe`, Oracle 23ai `IS DISTINCT FROM` (19c: `DECODE(a, b, 0, 1) = 1`).

## Q7. Freshness check  ·  🟢 easy

How fresh is `orders`? Show `latest_order_date` and `days_behind` = days from it to **2025-12-31** (pretend that is today). Show `status` = `STALE` if more than 2 days behind, else `FRESH`.

*Expected result: 1 row · columns: `latest_order_date`, `days_behind`, `status`*

**How it works:** Take `MAX(order_date)`, compute days to 'today', and label STALE when the gap is too big.

In [ ]:
%%oracle
SELECT MAX(order_date) AS latest_order_date,
       DATE '2025-12-31' - MAX(order_date) AS days_behind,
       CASE WHEN DATE '2025-12-31' - MAX(order_date) > 2 THEN 'STALE' ELSE 'FRESH' END AS status
FROM orders

,latest_order_date,days_behind,status
0,2025-12-28,3,STALE


**Why / traps:** In real checks use `CURRENT_DATE`. Freshness alerts catch an upstream job that silently stopped.

## Q8. Outliers  ·  🟢 easy

Show `order_id`, `customer_id`, `amount` of different orders whose amount is more than **3 times the average** amount.

*Expected result: 6 rows · columns: `order_id`, `customer_id`, `amount`*

**How it works:** Compute the overall average in a subquery and keep orders above 3 × that average.

In [ ]:
%%oracle
SELECT DISTINCT order_id, customer_id, amount
FROM orders
WHERE amount > 3 * (SELECT AVG(amount) FROM orders)

,order_id,customer_id,amount
0,1017,14,312.0
1,1019,16,306.0
2,1030,29,400.0
3,1049,28,405.0
4,1090,30,315.0
5,1095,30,315.0


## Q9. Values that fail a type cast  ·  🟡 medium

Raw files bring text. From the 4 values `'12'`, `'x7'`, `'3.5'`, `NULL`, show `raw_value` and `as_number` = the number, or NULL when the text is **not a number**. The query must not fail.

*Expected result: 4 rows · columns: `raw_value`, `as_number`*

**How it works:** A safe cast returns NULL instead of an error for bad text: `SAFE_CAST`, `TO_NUMBER(... DEFAULT NULL ON CONVERSION ERROR)`, `TRY_CAST`.

In [ ]:
%%oracle
SELECT raw_value, TO_NUMBER(raw_value DEFAULT NULL ON CONVERSION ERROR) AS as_number
FROM (SELECT '12' AS raw_value FROM dual UNION ALL
      SELECT 'x7' FROM dual UNION ALL
      SELECT '3.5' FROM dual UNION ALL
      SELECT NULL FROM dual)

,raw_value,as_number
0,12,12.0
1,x7,NULL
2,3.5,3.5
3,None,NULL


**Why / traps:** A bad row should not stop the whole load. Load it as NULL **and** count/route bad rows to an error table, so they are not silently lost.

`DEFAULT ... ON CONVERSION ERROR` exists from Oracle 12.2. `VALIDATE_CONVERSION(raw_value AS NUMBER)` returns 1/0.